# Extract Reproductive Annotation Summary

For each of the 20 batches in `sorted_batches_20_reprod_annot/`, this notebook:

1. Reads every JSON file in `with_reprod_annot/`.
2. Extracts all `Ri`, `N_R_i`, and `F_R_i` annotations from `corrected_text`.
3. For each annotation, extracts the surrounding **prose sentence** (from the closing `]` of the previous annotation to the `.` that ends the sentence containing the current annotation).
4. **Verifies** that the annotated value is present verbatim in the extracted prose.
5. Writes one summary JSON file per batch into `sorted_batches_20_reprod_annot/batch_XX/reprod_annot_summary.json`.

**Run cells in order: Cell 1 (config) → Cell 2 (helpers) → Cell 3 (verify) → Cell 4 (main loop) → Cell 5 (report).**

## Cell 1 — Imports and Configuration

In [1]:
import json
import re
from pathlib import Path

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATA_ROOT  = Path("../sorted_batches_20_reprod_annot")
N_BATCHES  = 20
OUTPUT_FILENAME = "reprod_annot_summary.json"

# Annotation pattern: captures entity type and value
# Matches [Ri = ...], [N_R_i = ...], [F_R_i = ...]
ANNOT_RE = re.compile(
    r'\[(?P<entity>Ri|N_R_i|F_R_i)\s*=\s*(?P<value>[^\]]+)\]'
)

print(f"Data root : {DATA_ROOT.resolve()}")
print(f"Batches   : {N_BATCHES}")
print(f"Output    : {OUTPUT_FILENAME}")
print(f"Pattern   : {ANNOT_RE.pattern}")

Data root : /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/sorted_batches_20_reprod_annot
Batches   : 20
Output    : reprod_annot_summary.json
Pattern   : \[(?P<entity>Ri|N_R_i|F_R_i)\s*=\s*(?P<value>[^\]]+)\]


## Cell 2 — Prose Extraction and Annotation Parsing

### Prose extraction rule

For each annotation `[Entity = value]` found in `corrected_text`:

- **Start**: immediately after the `]` of the *previous* annotation (or the start of the
  text if this is the first annotation).
- **End**: the first `.` that appears *after* the closing `]` of the current annotation.

If multiple target annotations fall within the same sentence (between the same pair of
start-boundary and sentence-ending `.`), the prose is extracted once and associated with
**all** of those annotations.

In [2]:
def extract_annotations(text: str) -> list[dict]:
    """
    Parse `text` and return one record per Ri / N_R_i / F_R_i annotation.

    Each record:
        {
            "entity":      "Ri" | "N_R_i" | "F_R_i",
            "annotation":  "[Ri = approximately 9.37 eggs per day]",
            "value":       "approximately 9.37 eggs per day",
            "prose":       "<sentence fragment containing the annotation>",
            "annot_start": <char offset of '[' in text>,
            "annot_end":   <char offset just past ']' in text>,
        }

    Prose boundaries:
      - start: position just after the ']' of the *previous* annotation
               found anywhere in the text (not only target entities), or 0.
      - end:   position of the first '.' after the ']' of *this* annotation,
               inclusive (so the '.' is included in the prose).

    All annotations that share the same sentence (same prose start / end pair)
    receive identical prose text.
    """
    # All bracket tokens in the text (any [...]) — used to track the previous ']'
    ANY_BRACKET = re.compile(r'\[[^\]]*\]')

    # Collect positions of every closing bracket in the text
    all_bracket_ends = [m.end() for m in ANY_BRACKET.finditer(text)]

    records = []

    for m in ANNOT_RE.finditer(text):
        entity     = m.group("entity")
        value      = m.group("value").strip()
        annotation = m.group(0)          # full "[Entity = value]"
        a_start    = m.start()
        a_end      = m.end()

        # --- prose start: just after the previous ']' anywhere in the text ---
        prev_ends = [e for e in all_bracket_ends if e <= a_start]
        if len(prev_ends) >= 2:
            # The last entry is the ']' of *this* annotation's own opening would
            # not exist yet; but all_bracket_ends includes the current match's end
            # only if it was processed. We want the ']' *before* '[' of this match.
            prose_start = prev_ends[-1]   # end of the bracket just before this one
        elif len(prev_ends) == 1:
            # The only prior bracket end is from the current annotation itself if
            # a_start == 0; otherwise it's a previous bracket.
            prose_start = prev_ends[0] if prev_ends[0] <= a_start else 0
        else:
            prose_start = 0

        # --- prose end: first '.' after the closing ']' of this annotation ---
        dot_pos = text.find(".", a_end)
        if dot_pos == -1:
            prose_end = len(text)
        else:
            prose_end = dot_pos + 1      # include the '.'

        prose = text[prose_start:prose_end].strip()

        records.append({
            "entity":      entity,
            "annotation":  annotation,
            "value":       value,
            "prose":       prose,
            "annot_start": a_start,
            "annot_end":   a_end,
        })

    return records


# ---------------------------------------------------------------------------
# Quick sanity check on the Abbottina rivularis example
# ---------------------------------------------------------------------------
_sample_path = (
    DATA_ROOT
    / "batch_01"
    / "with_reprod_annot"
    / "0002_Abbottina_rivularis_abj.json"
)
_sample = json.loads(_sample_path.read_text(encoding="utf-8"))
_recs   = extract_annotations(_sample["corrected_text"])

print(f"Annotations found: {len(_recs)}\n")
for r in _recs:
    print(f"  [{r['entity']}]  {r['annotation']}")
    print(f"  prose: {r['prose'][:120]}{'...' if len(r['prose']) > 120 else ''}")
    print()

Annotations found: 3

  [N_R_i]  [N_R_i = approximately 1711 eggs per nest]
  prose: , females typically construct nests and deposit approximately 1711 eggs per nest [N_R_i = approximately 1711 eggs per ne...

  [F_R_i]  [F_R_i = about 2 nesting events per year]
  prose: , a fecundity that is considerable given the modest adult body size. The same individuals are capable of completing abou...

  [Ri]  [Ri = approximately 9.37 eggs per day]
  prose: during the protracted spawning season, resulting in a daily reproduction rate of approximately 9.37 eggs per day [Ri = a...



## Cell 3 — Verification Logic

Two checks are performed for each annotation:

| Check | Pass condition |
|---|---|
| **value_in_prose** | The annotated value string appears verbatim (case-insensitive) inside the extracted prose. |
| **annotation_in_prose** | The full annotation bracket (e.g. `[Ri = approximately 9.37 eggs per day]`) appears verbatim inside the extracted prose. |

Both results are stored per record so that mismatches can be filtered and reviewed downstream.

In [3]:
def verify_annotation(record: dict) -> dict:
    """
    Add two verification flags to `record` (in-place) and return it.

    value_in_prose      : annotated value appears verbatim in prose (case-insensitive)
    annotation_in_prose : full bracket token appears verbatim in prose
    """
    prose      = record["prose"]
    value      = record["value"]
    annotation = record["annotation"]

    record["value_in_prose"]      = value.lower() in prose.lower()
    record["annotation_in_prose"] = annotation in prose
    return record


# ---------------------------------------------------------------------------
# Sanity-check on the Abbottina rivularis sample
# ---------------------------------------------------------------------------
print("Verification results for sample file:\n")
for r in _recs:
    verify_annotation(r)
    v_ok = "OK" if r["value_in_prose"]      else "FAIL"
    a_ok = "OK" if r["annotation_in_prose"] else "FAIL"
    print(f"  {r['entity']:8s}  value_in_prose={v_ok}  annotation_in_prose={a_ok}")
    print(f"           annotation : {r['annotation']}")
    print(f"           prose      : {r['prose'][:100]}...")
    print()

Verification results for sample file:

  N_R_i     value_in_prose=OK  annotation_in_prose=OK
           annotation : [N_R_i = approximately 1711 eggs per nest]
           prose      : , females typically construct nests and deposit approximately 1711 eggs per nest [N_R_i = approximat...

  F_R_i     value_in_prose=OK  annotation_in_prose=OK
           annotation : [F_R_i = about 2 nesting events per year]
           prose      : , a fecundity that is considerable given the modest adult body size. The same individuals are capabl...

  Ri        value_in_prose=OK  annotation_in_prose=OK
           annotation : [Ri = approximately 9.37 eggs per day]
           prose      : during the protracted spawning season, resulting in a daily reproduction rate of approximately 9.37 ...



## Cell 4 — Main Loop: Process All Batches

Iterates over all 20 batches, processes every JSON file in `with_reprod_annot/`,
and writes one `reprod_annot_summary.json` per batch directly into the batch folder.

### Output schema per batch summary file

```json
{
  "batch": "batch_01",
  "n_files": 318,
  "n_annotations": 954,
  "annotations": [
    {
      "species":              "Abbottina_rivularis",
      "entity":               "N_R_i",
      "annotation":           "[N_R_i = approximately 1711 eggs per nest]",
      "prose":                "...",
      "value_in_prose":       true,
      "annotation_in_prose":  true
    },
    ...
  ]
}
```

In [4]:
grand_files       = 0
grand_annotations = 0
grand_verify_fail = 0

batch_stats = []   # collected for the report cell

for batch_num in range(1, N_BATCHES + 1):
    batch_name = f"batch_{batch_num:02d}"
    src_dir    = DATA_ROOT / batch_name / "with_reprod_annot"

    if not src_dir.exists():
        print(f"[WARN] {src_dir} not found — skipping")
        continue

    json_files = sorted(src_dir.glob("*.json"))
    annotations_out = []
    n_verify_fail   = 0

    for fp in json_files:
        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            print(f"  [PARSE-ERROR] {fp.name}")
            continue

        species = data.get("specie", fp.stem)
        text    = data.get("corrected_text", "")
        if not text:
            continue

        for rec in extract_annotations(text):
            verify_annotation(rec)

            if not rec["value_in_prose"] or not rec["annotation_in_prose"]:
                n_verify_fail += 1

            annotations_out.append({
                "species":             species,
                "entity":              rec["entity"],
                "annotation":          rec["annotation"],
                "prose":               rec["prose"],
                "value_in_prose":      rec["value_in_prose"],
                "annotation_in_prose": rec["annotation_in_prose"],
            })

    summary = {
        "batch":         batch_name,
        "n_files":       len(json_files),
        "n_annotations": len(annotations_out),
        "annotations":   annotations_out,
    }

    out_path = DATA_ROOT / batch_name / OUTPUT_FILENAME
    out_path.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    grand_files       += len(json_files)
    grand_annotations += len(annotations_out)
    grand_verify_fail += n_verify_fail

    batch_stats.append({
        "batch":        batch_name,
        "n_files":      len(json_files),
        "n_annot":      len(annotations_out),
        "n_fail":       n_verify_fail,
    })

    fail_note = f"  *** {n_verify_fail} verification failures" if n_verify_fail else ""
    print(
        f"{batch_name}: {len(json_files):4d} files | "
        f"{len(annotations_out):5d} annotations | "
        f"written → {out_path.name}"
        + fail_note
    )

print(
    f"\nGrand total: {grand_files} files | "
    f"{grand_annotations} annotations | "
    f"{grand_verify_fail} verification failures"
)

batch_01:  318 files |   698 annotations | written → reprod_annot_summary.json
batch_02:  313 files |   662 annotations | written → reprod_annot_summary.json
batch_03:  323 files |   704 annotations | written → reprod_annot_summary.json
batch_04:  300 files |   697 annotations | written → reprod_annot_summary.json
batch_05:  325 files |   711 annotations | written → reprod_annot_summary.json
batch_06:  303 files |   650 annotations | written → reprod_annot_summary.json
batch_07:  314 files |   686 annotations | written → reprod_annot_summary.json
batch_08:  329 files |   723 annotations | written → reprod_annot_summary.json
batch_09:  307 files |   681 annotations | written → reprod_annot_summary.json
batch_10:  306 files |   667 annotations | written → reprod_annot_summary.json
batch_11:  320 files |   703 annotations | written → reprod_annot_summary.json
batch_12:  328 files |   746 annotations | written → reprod_annot_summary.json
batch_13:  296 files |   652 annotations | written →

## Cell 5 — Summary Report

Prints a per-batch table and lists any annotation records that failed verification.

In [5]:
# ---------------------------------------------------------------------------
# Per-batch table
# ---------------------------------------------------------------------------
header = f"{'Batch':<12} {'Files':>6} {'Annots':>8} {'Failures':>10}  {'Output'}"
print(header)
print("-" * len(header))

total_files = total_annot = total_fail = 0
for s in batch_stats:
    fail_flag = " <-- REVIEW" if s["n_fail"] else ""
    print(
        f"{s['batch']:<12} {s['n_files']:>6} {s['n_annot']:>8} "
        f"{s['n_fail']:>10}  {OUTPUT_FILENAME}{fail_flag}"
    )
    total_files += s["n_files"]
    total_annot += s["n_annot"]
    total_fail  += s["n_fail"]

print("-" * len(header))
print(f"{'TOTAL':<12} {total_files:>6} {total_annot:>8} {total_fail:>10}")

# ---------------------------------------------------------------------------
# Detail view of failed records (load back from written files)
# ---------------------------------------------------------------------------
if total_fail:
    print(f"\n{'='*70}")
    print(f"FAILED RECORDS ({total_fail} total)")
    print(f"{'='*70}\n")

    for s in batch_stats:
        if s["n_fail"] == 0:
            continue
        summary_path = DATA_ROOT / s["batch"] / OUTPUT_FILENAME
        summary_data = json.loads(summary_path.read_text(encoding="utf-8"))
        failures = [
            a for a in summary_data["annotations"]
            if not a["value_in_prose"] or not a["annotation_in_prose"]
        ]
        print(f"--- {s['batch']} ({len(failures)} failures) ---")
        for f in failures:
            print(f"  species    : {f['species']}")
            print(f"  entity     : {f['entity']}")
            print(f"  annotation : {f['annotation']}")
            print(f"  value_in_prose      : {f['value_in_prose']}")
            print(f"  annotation_in_prose : {f['annotation_in_prose']}")
            print(f"  prose      : {f['prose'][:200]}{'...' if len(f['prose']) > 200 else ''}")
            print()
else:
    print("\nAll annotations passed verification.")

Batch         Files   Annots   Failures  Output
-----------------------------------------------
batch_01        318      698          0  reprod_annot_summary.json
batch_02        313      662          0  reprod_annot_summary.json
batch_03        323      704          0  reprod_annot_summary.json
batch_04        300      697          0  reprod_annot_summary.json
batch_05        325      711          0  reprod_annot_summary.json
batch_06        303      650          0  reprod_annot_summary.json
batch_07        314      686          0  reprod_annot_summary.json
batch_08        329      723          0  reprod_annot_summary.json
batch_09        307      681          0  reprod_annot_summary.json
batch_10        306      667          0  reprod_annot_summary.json
batch_11        320      703          0  reprod_annot_summary.json
batch_12        328      746          0  reprod_annot_summary.json
batch_13        296      652          0  reprod_annot_summary.json
batch_14        321      708     